# Задание 1

## Часть 1

Напишите такой класс Animal, чтобы при вызове его классового метода get_highest() возвращался рост самого высокого из экземпляров

In [29]:
class Animal():
  highest = None

  def __init__(self, name, height):
    self.name = name
    self.height = height
    if Animal.highest is None or Animal.highest.height < height:
      Animal.highest = self

  def speak(self):
    print("...")

  def __eq__(self, other):
    return type(self) is type(other)

  @classmethod
  def get_highest(cls):
    return cls.highest.height

In [30]:
a1 = Animal("Alex", 200)
a2 = Animal("Melman", 400)
a3 = Animal("King Julien XIII", 40)
a4 = Animal("Maurice", 40)
assert Animal.get_highest() == 400

## Часть 2

Все же считать всех животных просто.. животными это странно. Давайте заведем каждому животному по своему классу, унаследовавшись от класса Animal, при этом сделайте так, чтобы при сравнении двух животных только для животных одного вида возвращалось True

In [31]:
class Lion(Animal):
  def __init__(self, name, height):
    super().__init__(name, height)

class Lemur(Animal):
  def __init__(self, name, height):
    super().__init__(name, height)

class Giraffe(Animal):
  def __init__(self, name, height):
    super().__init__(name, height)

In [ ]:
a1 = Lion("Alex", 200)
a2 = Giraffe("Melman", 400)
a3 = Lemur("King Julien XIII", 40)
a4 = Lemur("Maurice", 40)

#переписала правило для метода __eq__ в родительском классе
assert a4 == a3
assert a1 != a2
assert a2 != a4

## Часть 3

Это все здорово, но как-то неправильно, что класс Animal имеет метод speak, а классы-потомки не определяют свою реализацию. Давайте сделаем так, чтобы все виды животных были обязаны иметь свою реализацию метода speak, но просто animal'ов уже нельзя было создать

In [39]:
from abc import ABC, abstractmethod

class Animal(ABC):

  highest = None 

  def __init__(self, name, height):
    self.name = name
    self.height = height
    if Animal.highest is None or Animal.highest.height < height:
      Animal.highest = self

  @abstractmethod
  def speak(self):
    pass

  def __eq__(self, other):
    return type(self) is type(other)

  @classmethod
  def get_highest(cls):
    return cls.highest.height

class Lion(Animal):
  def speak(self):
    print("Я лев")

class Lemur(Animal):
  def speak(self):
    print("Я лемур")

class Giraffe(Animal):
  def speak(self):
    print("Я жираф")

In [40]:
import unittest
from contextlib import contextmanager

class AnimalTestCase(unittest.TestCase):
  @contextmanager
  def assertNotRaises(self, exc_type):
    print(0)
    try:
      yield None
    except exc_type:
      raise self.failureException('{} raised'.format(exc_type.__name__))

  def test_create_animal_instance(self):
    with self.assertRaises(TypeError):
      a = Animal("", 0)

  def test_create_zebra(self):
    with self.assertRaises(TypeError):
      class Zebra(Animal):
        pass
      a = Zebra("", 0)

  def test_create_animals(self):
    with self.assertNotRaises(TypeError):
      a1 = Lion("Alex", 200)
      a2 = Giraffe("Melman", 400)
      a3 = Lemur("King Julien XIII", 40)
      a4 = Lemur("Maurice", 40)
      a1.speak()
      a2.speak()
      a3.speak()
      a4.speak()

unittest.main(argv=[''], verbosity=0, exit=False)

0
Я лев
Я жираф
Я лемур
Я лемур


----------------------------------------------------------------------
Ran 3 tests in 0.000s

OK


Вы должны увидеть такой результат

0
----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK
<unittest.main.TestProgram at 0x7f081faf2190>

# Задание 2

## Исходные данные

Представим ситуацию, что вы нашли где-то целый архив с файлами переводов и решили обработать эти файлы.  Проблема в том, что все файлы имеют разные форматы:

In [15]:
#.xliff
file1 = '''
<?xml version="1.0" encoding="UTF-8"?>
<xliff version="1.2"
  xmlns="urn:oasis:names:tc:xliff:document:1.2"
  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
  xsi:schemaLocation="urn:oasis:names:tc:xliff:document:1.2 xliff-core-1.2-strict.xsd">
<file original="sample.properties"
  source-language="en" target-language="de-AT"
  datatype="javapropertyresourcebundle">
<body>
   <trans-unit id="1" resname="welcome">
      <source xml:lang="en">Hooray, you're here! The day just got better - enjoy the following tips!</source>
      <target xml:lang="de-AT">Hurra, du bist hier! Der Tag ist gerettet - nutze die folgenden Tipps!</target>
   </trans-unit>
   <trans-unit id="2" resname="text_segment">
      <source xml:lang="en">A section of text like this is known as a text segment. Start rockin' your translations now!</source>
      <target xml:lang="de-AT">Eine Textpassage wie diese bezeichnen wir als Textsegment. Starte jetzt mit deinen Übersetzungen durch!</target>
   </trans-unit>
   <trans-unit id="3" resname="tab_shortcut">
      <source xml:lang="en">Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done.</source>
      <target xml:lang="de-AT">Arriba, Arriba! Andale, Andale! Sei schneller als Speedy Gonzales. Springe mit TAB ins nächste Textsegment, sobald du fertig bist. Deine Änderungen werden automatisch gespeichert.</target>
   </trans-unit>
   <trans-unit id="4" resname="love">
      <source xml:lang="en">Made with ❤</source>
      <target xml:lang="de-AT">❤</target>
      <note>lh-check { min: 10, max: 15 }</note>
   </trans-unit>
</body>
</file>
</xliff>
'''

#.po
file2 = '''
msgid ""
msgstr ""
"Project-Id-Version: LingoHub 2.2.4\n"
"Report-Msgid-Bugs-To: \n"
"POT-Creation-Date: 2015-03-05 19:22+0200\n"
"PO-Revision-Date: 2015-04-29 10:16+0100\n"
"Last-Translator: Translator <translator@lingohub.com>\n"
"MIME-Version: 1.0\n"
"Content-Type: text/plain; charset=UTF-8\n"
"Content-Transfer-Encoding: 8bit\n"
"Plural-Forms: nplurals=2; plural=n>1;\n"

msgid "Hooray, you're here! The day just got better - enjoy the following tips!"
msgstr "Hurra, du bist hier! Der Tag ist gerettet - nutze die folgenden Tipps!"

msgid "A section of text like this is known as a text segment. Start rockin' your translations now!"
msgstr "Eine Textpassage wie diese bezeichnen wir als Textsegment. Starte jetzt mit deinen Übersetzungen durch!"

msgid "Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done."
msgstr "Arriba, Arriba! Andale, Andale! Sei schneller als Speedy Gonzales. Springe mit TAB ins nächste Textsegment, sobald du fertig bist. Deine Änderungen werden automatisch gespeichert."

msgid "Made with ❤"
msgstr "❤"
'''

#.po
file3 = '''
msgid ""
msgstr ""
"Project-Id-Version: LingoHub 2.2.4\n"
"Report-Msgid-Bugs-To: \n"
"POT-Creation-Date: 2015-03-05 19:22+0200\n"
"PO-Revision-Date: 2015-04-29 10:16+0100\n"
"Last-Translator: Translator <translator@lingohub.com>\n"
"MIME-Version: 1.0\n"
"Content-Type: text/plain; charset=UTF-8\n"
"Content-Transfer-Encoding: 8bit\n"
"Plural-Forms: nplurals=2; plural=n>1;\n"

msgid "Hooray, you're here! The day just got better - enjoy the following tips!"
msgstr "Hurrah, tu es là. Ça fait une bonne journée - profite des renseignements suivants !"

msgid "A section of text like this is known as a text segment. Start rockin' your translations now!"
msgstr "Un tel extrait est qualifié de segment du texte chez nous. C’est parti – commence tes traductions tout de suite/maintenant !"

msgid "Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done."
msgstr "Arriba, Arriba ! Andale, Andale ! Prends Speedy Gonzales de vitesse. Appuie sur TAB pour sauvegarder et continuer avec le segment prochain dès que tu as fini."

msgid "We just like to see you happy, that's why LingoChecks automatically check translations for predetermined criteria.\n Among other checks LingoHub verifies if %s used in the original text are also present in translated texts."
msgstr "Bazinga !"

msgid "Made with ❤"
msgstr "❤"
'''

#.ts
file4 = '''
<?xml version="1.0" encoding="utf-8"?>
<!DOCTYPE TS>
<TS version="2.1" language="de_AT">
<context>
    <name>examples</name>
    <message>
        <source>Hooray, you're here! The day just got better - enjoy the following tips!</source>
        <translation>Hurra, du bist hier! Der Tag ist gerettet - nutze die folgenden Tipps!</translation>
    </message>
    <message>
        <source>A section of text like this is known as a text segment. Start rockin' your translations now!</source>
	      <translation>Eine Textpassage wie diese bezeichnen wir als Textsegment. Starte jetzt mit deinen Übersetzungen durch!</translation>
    </message>
    <message>
        <source>Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done.</source>
	      <translation>Arriba, Arriba! Andale, Andale! Sei schneller als Speedy Gonzales. Springe mit TAB ins nächste Textsegment, sobald du fertig bist. Deine Änderungen werden automatisch gespeichert.</translation>
    </message>
    <message>
        <source>Made with ❤</source>
        <translation>❤</translation>
        <comment>lh-check { min: 10, max: 15 }</comment>
    </message>
</context>
</TS>
'''

#.ts
file5 = '''
<?xml version="1.0" encoding="utf-8"?>
<!DOCTYPE TS>
<TS version="2.1" language="fr">
<context>
    <name>examples</name>
    <message>
        <source>Hooray, you're here! The day just got better - enjoy the following tips!</source>
        <translation>Hurrah, tu es là. Ça fait une bonne journée - profite des renseignements suivants !</translation>
    </message>
    <message>
        <source>A section of text like this is known as a text segment. Start rockin' your translations now!</source>
	      <translation>Un tel extrait est qualifié de segment du texte chez nous. C’est parti – commence tes traductions tout de suite/maintenant !</translation>
    </message>
    <message>
        <source>Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done.</source>
	      <translation>Arriba, Arriba ! Andale, Andale ! Prends Speedy Gonzales de vitesse. Appuie sur TAB pour sauvegarder et continuer avec le segment prochain dès que tu as fini.</translation>
    </message>
    <message>
        <source>Made with ❤</source>
        <translation>❤</translation>
        <comment>lh-check { min: 10, max: 15 }</comment>
    </message>
</context>
</TS>
'''

## Часть 1

Мы хотим:
- уметь распарсить файлы разного вида
- уметь переводить файлы из одного формата в другой
- для тех форматов, где язык не указан, хотим вызывать какой-то метод/функцию для определения языка
- уметь собирать статистику над файлами всех форматов (что-то типа "а посчитайте мне, сколько раз среди заданного куска массива с файлами переводов какой язык встречается")

In [17]:
pip install langdetect

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 7.2 MB/s  0:00:00eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993332 sha256=5e0dffad4f329ab59c4deee060cf765dc172f2a4f8f6e5efed652c433c63f712
  Stored in directory: /Users/kulyatinakatya/Library/Caches/pip/wheels/d1/c1/d9/7e068de779d863bc8f8fc9467d85e25cfe47fa5051fff1a1bb
Successfully built langdetect

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import re
import xml.etree.ElementTree as ET
from langdetect import detect


class Format1(): #.xliff
  def __init__(self):
      self.pairs = []       #список для пар фраз (транс-юнитов) на двух языках типа [(source1, target1), (source2, target2), ...]
      self.source_language = None
      self.target_language = None

  @classmethod
  def parse(cls, text):
    text = text.strip()
    ns = {"xliff": "urn:oasis:names:tc:xliff:document:1.2"}
    root = ET.fromstring(text)
    f = root.find("xliff:file", ns)

    obj = cls()
    obj.source_language = f.get("source-language")
    obj.target_language = f.get("target-language")

    for tu in f.iterfind(".//xliff:trans-unit", ns):
      s = tu.find("xliff:source", ns)
      t = tu.find("xliff:target", ns)
      obj.pairs.append(((s.text or "") if s is not None else "", (t.text or "") if t is not None else ""))
    
    return obj

  def convert(self):
        output = [
            '<?xml version="1.0" encoding="UTF-8"?>',
            '<xliff version="1.2" '
            'xmlns="urn:oasis:names:tc:xliff:document:1.2">',
            f'<file source-language="{self.source_language or "en"}" '
            f'target-language="{self.target_language or ""}">',
            '<body>',
        ]
        for i, (s, t) in enumerate(self.pairs, 1):
            output += [
                f'<trans-unit id="{i}">',
                f'<source>{s}</source>',
                f'<target>{t}</target>',
                '</trans-unit>',
            ]
        output += ['</body>', '</file>', '</xliff>']
        return "\n".join(output)

  def ensure_language(self): #вспомогательная функция для определения языка
    if self.target_language:
      return self.target_language
    sample = " ".join(t for _, t in self.pairs if t)
    if sample:
      self.target_language = detect_language(sample)
    return self.target_language

class Format2(): #.po
  def __init__(self):
      self.pairs = []       #список для пар фраз (транс-юнитов) на двух языках типа [(source1, target1), (source2, target2), ...]
      self.source_language = None
      self.target_language = None

  @classmethod
  def parse(cls, text):
    obj = cls()

    m = re.search(r'"Language:\s*([^\\"]*)\\n"', text)
    if m:
      obj.target_language = m.group(1)

    msg_pattern = r'msgid\s+"((?:[^"\\]|\\.)*)"\s*\nmsgstr\s+"((?:[^"\\]|\\.)*)"'
    for s, t in re.findall(msg_pattern, text):
      if s:   
        obj.pairs.append((s, t))

    return obj

  def convert(self):
    output = ['msgid ""', 'msgstr ""', f'"Language: {self.target_language or ""}\\n"', '']
    for s, t in self.pairs:
      output += [f'msgid "{s}"', f'msgstr "{t}"', '']
    return "\n".join(output)

  def ensure_language(self): #вспомогательная функция для определения языка
    if self.target_language:
      return self.target_language
    sample = " ".join(t for _, t in self.pairs if t)
    if sample:
      self.target_language = detect_language(sample)
    return self.target_language


class Format3(): #.ts
  def __init__(self):
      self.pairs = []       #список для пар фраз (транс-юнитов) на двух языках типа [(source1, target1), (source2, target2), ...]
      self.source_language = None
      self.target_language = None

  @classmethod
  def parse(cls, text):
    obj = cls()
    text = text.strip()
    root = ET.fromstring(text)
    obj.target_language = root.get("language")

    for msg in root.iter("message"):
      s = msg.find("source")
      t = msg.find("translation")
      obj.pairs.append(((s.text or "") if s is not None else "", (t.text or "") if t is not None else ""))

    return obj

  def convert(self):
    output = [
          '<?xml version="1.0" encoding="utf-8"?>',
          '<!DOCTYPE TS>',
          f'<TS version="2.1" language="{self.target_language or ""}">',
          '<context><name>examples</name>',
        ]
    for s, t in self.pairs:
      output += ['<message>',
             f'<source>{s}</source>',
             f'<translation>{t}</translation>',
            '</message>']
    output += ['</context>', '</TS>']
    return "\n".join(output)

  def ensure_language(self): #вспомогательная функция для определения языка
    if self.target_language:
      return self.target_language
    sample = " ".join(t for _, t in self.pairs if t)
    if sample:
      self.target_language = detect_language(sample)
    return self.target_language


def get_converted(s_obj, trg_cls):
    s_obj.ensure_language() #идем во вспомогательную функцию для опеределения есть язык или нет (и его доопределения)
    obj = trg_cls()
    obj.pairs = list(s_obj.pairs)
    obj.source_language = s_obj.source_language
    obj.target_language = s_obj.target_language
    return obj

def parse_file(text):
    if "<xliff" in text:
        return Format1.parse(text)
    if "<TS" in text:
        return Format3.parse(text)
    return Format2.parse(text)

def detect_language(text):
    try:
        return detect(text)
    except Exception:
        return "unknown"

def normalize_lang(code): #эта функция для того, чтобы избежать расхождения в языках типа de/de-AT/de_AT
    if not code:
        return None
    return code.replace("_", "-").lower().split("-")[0]

def stats(ff):
  result = {"files": 0, "formats": {}, "languages": {}, "pairs": 0}
  for f in ff:
      f.ensure_language()
      result["files"] += 1
      fmt = type(f).__name__
      result["formats"][fmt] = result["formats"].get(fmt, 0) + 1
      lang = normalize_lang(f.target_language or "unknown")
      result["languages"][lang] = result["languages"].get(lang, 0) + 1
      result["pairs"] += len(f.pairs)
  return result


In [29]:
files = [parse_file(t) for t in (file1, file2, file3, file4, file5)] #все файлики загружаем, определяем формат, передаем в класс и парсим


print(get_converted(files[0], Format2).convert())   #тут например переводим из .xliff в .po

msgid ""
msgstr ""
"Language: de-AT\n"

msgid "Hooray, you're here! The day just got better - enjoy the following tips!"
msgstr "Hurra, du bist hier! Der Tag ist gerettet - nutze die folgenden Tipps!"

msgid "A section of text like this is known as a text segment. Start rockin' your translations now!"
msgstr "Eine Textpassage wie diese bezeichnen wir als Textsegment. Starte jetzt mit deinen Übersetzungen durch!"

msgid "Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done."
msgstr "Arriba, Arriba! Andale, Andale! Sei schneller als Speedy Gonzales. Springe mit TAB ins nächste Textsegment, sobald du fertig bist. Deine Änderungen werden automatisch gespeichert."

msgid "Made with ❤"
msgstr "❤"



In [30]:
print(stats(files)) #получаем статистику по корпусу (например, как в задании, считаем кол-во документов на каждом из языков)

{'files': 5, 'formats': {'Format1': 1, 'Format2': 2, 'Format3': 2}, 'languages': {'de': 3, 'fr': 2}, 'pairs': 21}


In [ ]:
#теперь сделаем все через класс универсальный формат, от которого будут наследоваться другие классы
#я тут перестроила чуть структуру, но смысл и демонстративные запросы остались те же

import re
import xml.etree.ElementTree as ET
from langdetect import detect


def normalize_lang(code):
    if not code:
        return None
    return code.replace("_", "-").lower().split("-")[0]


def detect_language(text):
    try:
        return detect(text)
    except Exception:
        return "unknown"


class UniversalFormat:
    def __init__(self):
        self.pairs = []            
        self.source_language = None
        self.target_language = None

    def ensure_language(self):
        if self.target_language:
            self.target_language = normalize_lang(self.target_language)
            return self.target_language
        sample = " ".join(t for _, t in self.pairs if t)
        if sample:
            self.target_language = normalize_lang(detect_language(sample))
        return self.target_language

    @classmethod
    def parse(cls, text):
        raise NotImplementedError

    def convert(self):
        raise NotImplementedError


class Format1(UniversalFormat): #.xliff
  @classmethod
  def parse(cls, text):
    text = text.strip()
    ns = {"xliff": "urn:oasis:names:tc:xliff:document:1.2"}
    root = ET.fromstring(text)
    f = root.find("xliff:file", ns)

    obj = cls()
    obj.source_language = f.get("source-language")
    obj.target_language = f.get("target-language")

    for tu in f.iterfind(".//xliff:trans-unit", ns):
      s = tu.find("xliff:source", ns)
      t = tu.find("xliff:target", ns)
      obj.pairs.append(((s.text or "") if s is not None else "", (t.text or "") if t is not None else ""))
    
    return obj

  def convert(self):
        output = [
            '<?xml version="1.0" encoding="UTF-8"?>',
            '<xliff version="1.2" '
            'xmlns="urn:oasis:names:tc:xliff:document:1.2">',
            f'<file source-language="{self.source_language or "en"}" '
            f'target-language="{self.target_language or ""}">',
            '<body>',
        ]
        for i, (s, t) in enumerate(self.pairs, 1):
            output += [
                f'<trans-unit id="{i}">',
                f'<source>{s}</source>',
                f'<target>{t}</target>',
                '</trans-unit>',
            ]
        output += ['</body>', '</file>', '</xliff>']
        return "\n".join(output)


class Format2(UniversalFormat): #.po
  @classmethod
  def parse(cls, text):
    obj = cls()

    m = re.search(r'"Language:\s*([^\\"]*)\\n"', text)
    if m:
      obj.target_language = m.group(1)

    msg_pattern = r'msgid\s+"((?:[^"\\]|\\.)*)"\s*\nmsgstr\s+"((?:[^"\\]|\\.)*)"'
    for s, t in re.findall(msg_pattern, text):
      if s:   
        obj.pairs.append((s, t))

    return obj

  def convert(self):
    output = ['msgid ""', 'msgstr ""', f'"Language: {self.target_language or ""}\\n"', '']
    for s, t in self.pairs:
      output += [f'msgid "{s}"', f'msgstr "{t}"', '']
    return "\n".join(output)


class Format3(UniversalFormat): #.ts
  @classmethod
  def parse(cls, text):
    obj = cls()
    text = text.strip()
    root = ET.fromstring(text)
    obj.target_language = root.get("language")

    for msg in root.iter("message"):
      s = msg.find("source")
      t = msg.find("translation")
      obj.pairs.append(((s.text or "") if s is not None else "", (t.text or "") if t is not None else ""))

    return obj

  def convert(self):
    output = [
          '<?xml version="1.0" encoding="utf-8"?>',
          '<!DOCTYPE TS>',
          f'<TS version="2.1" language="{self.target_language or ""}">',
          '<context><name>examples</name>',
        ]
    for s, t in self.pairs:
      output += ['<message>',
             f'<source>{s}</source>',
             f'<translation>{t}</translation>',
            '</message>']
    output += ['</context>', '</TS>']
    return "\n".join(output)


def get_converted(s_obj, trg_cls):
    s_obj.ensure_language()
    obj = trg_cls()
    obj.pairs = list(s_obj.pairs)
    obj.source_language = s_obj.source_language
    obj.target_language = s_obj.target_language
    return obj


def parse_file(text):
    if "<xliff" in text:
        return Format1.parse(text)
    if "<TS" in text:
        return Format3.parse(text)
    return Format2.parse(text)


def stats(ff):
    result = {"files": 0, "formats": {}, "languages": {}, "pairs": 0}
    for f in ff:
        f.ensure_language()
        result["files"] += 1
        fmt = type(f).__name__
        result["formats"][fmt] = result["formats"].get(fmt, 0) + 1
        lang = normalize_lang(f.target_language) or "unknown"
        result["languages"][lang] = result["languages"].get(lang, 0) + 1
        result["pairs"] += len(f.pairs)
    return result


def count_language(ff, lang):
    lang = normalize_lang(lang)
    n = 0
    for f in ff:
        f.ensure_language()
        if f.target_language == lang:
            n += 1
    return n


In [ ]:
#те же тесты
files = [parse_file(t) for t in (file1, file2, file3, file4, file5)] #все файлики загружаем, определяем формат, передаем в класс и парсим
print(get_converted(files[0], Format2).convert())   #тут например переводим из .xliff в .po
print(stats(files)) #получаем статистику по корпусу (например, как в задании, считаем кол-во документов на каждом из языков)

msgid ""
msgstr ""
"Language: de\n"

msgid "Hooray, you're here! The day just got better - enjoy the following tips!"
msgstr "Hurra, du bist hier! Der Tag ist gerettet - nutze die folgenden Tipps!"

msgid "A section of text like this is known as a text segment. Start rockin' your translations now!"
msgstr "Eine Textpassage wie diese bezeichnen wir als Textsegment. Starte jetzt mit deinen Übersetzungen durch!"

msgid "Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done."
msgstr "Arriba, Arriba! Andale, Andale! Sei schneller als Speedy Gonzales. Springe mit TAB ins nächste Textsegment, sobald du fertig bist. Deine Änderungen werden automatisch gespeichert."

msgid "Made with ❤"
msgstr "❤"

{'files': 5, 'formats': {'Format1': 1, 'Format2': 2, 'Format3': 2}, 'languages': {'de': 3, 'fr': 2}, 'pairs': 21}


## Часть 2

Мы нашли еще данных, но уже в других форматах:

In [ ]:
# .csv
file6 = '''
"Title","Description","en", "de-AT"
"welcome",,"Hooray, you're here! The day just got better - enjoy the following tips!", "Hurra, du bist hier! Der Tag ist gerettet - nutze die folgenden Tipps!"
"text_segment",,"A section of text like this is known as a text segment. Start rockin' your translations now!", "Eine Textpassage wie diese bezeichnen wir als Textsegment. Starte jetzt mit deinen Übersetzungen durch!"
"tab_shortcut",,"Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done.", "Arriba, Arriba! Andale, Andale! Sei schneller als Speedy Gonzales. Springe mit TAB ins nächste Textsegment, sobald du fertig bist. Deine Änderungen werden automatisch gespeichert."
"love","lh-check { max: 15, min: 10 }","Made with ❤", "❤"
'''

# .resw
file7_1 = '''
<?xml version="1.0" encoding="utf-8"?>
<root>
  <data name="welcome" xml:space="preserve">
    <value>Hooray, you're here! The day just got better - enjoy the following tips!</value>
  </data>
  <data name="text_segment" xml:space="preserve">
    <value>A section of text like this is known as a text segment. Start rockin' your translations now!</value>
  </data>
  <data name="tab_shortcut" xml:space="preserve">
    <value>Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done.</value>
  </data>
  <data name="love">
    <value>Made with ❤</value>
    <comment>lh-check { min: 10, max: 15 }</comment>
  </data>
</root>
'''

file7_2 = '''
<?xml version="1.0" encoding="utf-8"?>
<root>
  <data name="welcome" xml:space="preserve">
    <value>Hurrah, tu es là. Ça fait une bonne journée - profite des renseignements suivants !</value>
  </data>
  <data name="text_segment" xml:space="preserve">
    <value>Un tel extrait est qualifié de segment du texte chez nous. C’est parti – commence tes traductions tout de suite/maintenant !</value>
  </data>
  <data name="tab_shortcut" xml:space="preserve">
    <value>Arriba, Arriba ! Andale, Andale ! Prends Speedy Gonzales de vitesse. Appuie sur TAB pour sauvegarder et continuer avec le segment prochain dès que tu as fini.</value>
  </data>
  <data name="love" xml:space="preserve">
    <value>❤</value>
  </data>
</root>
'''

# .php
file8_1 = '''
<?php
  define("welcome", "Hooray, you're here! The day just got better - enjoy the following tips!");

  define("text_segment", "A section of text like this is known as a text segment. Start rockin' your translations now!");

  define("tab_shortcut", "Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done.");

  # lh-check { min: 10, max: 15 }
  define("love", "Made with ❤");
?>
'''

file8_2 = '''
<?php
  define("welcome", "Hurra, du bist hier! Der Tag ist gerettet - nutze die folgenden Tipps!");

  define("text_segment", "Eine Textpassage wie diese bezeichnen wir als Textsegment. Starte jetzt mit deinen Übersetzungen durch!");

  define("tab_shortcut", "Arriba, Arriba! Andale, Andale! Sei schneller als Speedy Gonzales. Springe mit TAB ins nächste Textsegment, sobald du fertig bist. Deine Änderungen werden automatisch gespeichert.");

  define("love", "❤");
?>
'''

Как именно изменится ваша модель классов для того, чтобы поддержать новые форматы?

In [ ]:
#добавится еще три класса Format4, Format5, Format6, в которых надо прописать структуру соответствующих файлов

#+надо  в функции parse_file добавить флажки для новых типов, то есть она будет выглядеть как-то так:

def parse_file(text):
    s = text.lstrip()
    if "<xliff" in text:  return Format1.parse(text)
    if "<TS" in text:     return Format3.parse(text)
    if "<root>" in text:  return Format5.parse(text)   # .resw
    if s.startswith("<?php"): return Format6.parse(text)  # .php
    if s.startswith('"'): return Format4.parse(text)   # .csv
    return Format2.parse(text)      
